Para melhor visualização dos csvs

In [2]:
import requests
import pandas as pd
import os

In [3]:
ANO_REFERENCIA = 2024

url_votos = f'https://dadosabertos.camara.leg.br/arquivos/votacoesVotos/csv/votacoesVotos-{ANO_REFERENCIA}.csv'
url_votacoes = f'https://dadosabertos.camara.leg.br/arquivos/votacoes/csv/votacoes-{ANO_REFERENCIA}.csv'

In [ ]:
from pathlib import Path

def encontrar_raiz(marcador="requirements.txt"):
    caminho = Path.cwd()
    while not (caminho / marcador).exists():
        caminho = caminho.parent
    return caminho

RAIZ = encontrar_raiz()
pasta = RAIZ / "dados" / "brutos"
pasta.mkdir(parents=True, exist_ok=True)

for url in [url_votos, url_votacoes]:
    nome_arquivo = url.split('/')[-1]
    caminho_arquivo = pasta / nome_arquivo

    if not caminho_arquivo.exists():
        print(f'Baixando {nome_arquivo}...')
        response = requests.get(url)
        with open(caminho_arquivo, 'wb') as f:
            f.write(response.content)
    else:
        print(f'{nome_arquivo} já existe. Pulando download.')

with open(pasta / f'votacoesVotos-{ANO_REFERENCIA}.csv', 'r', encoding='utf-8') as f:
    print(f.readline())

votacoesVotos-2024.csv já existe. Pulando download.
votacoes-2024.csv já existe. Pulando download.
﻿"idVotacao";"uriVotacao";"dataHoraVoto";"voto";"deputado_id";"deputado_uri";"deputado_nome";"deputado_siglaPartido";"deputado_uriPartido";"deputado_siglaUf";"deputado_idLegislatura";"deputado_urlFoto"



In [10]:
df = pd.read_csv(pasta / f'votacoesVotos-{ANO_REFERENCIA}.csv', sep=';', encoding='utf-8-sig')

In [11]:
colunas_uteis = ['idVotacao', 'deputado_id', 'voto', 'deputado_siglaPartido', 'deputado_siglaUf']
df_limpo = df[colunas_uteis]

(RAIZ / 'dados' / 'processado').mkdir(parents=True, exist_ok=True)
df_limpo.to_csv(RAIZ / 'dados' / 'processado' / f'votos-{ANO_REFERENCIA}-limpo.csv', index=False)


In [12]:
deputados_df = df.drop_duplicates(subset='deputado_id')[
    ['deputado_id', 'deputado_nome', 'deputado_siglaPartido', 'deputado_siglaUf', 'deputado_urlFoto']
].copy()

deputados_df.to_csv(RAIZ / 'dados' / 'processado' / 'deputados.csv', index=False)